# Verification Notebook V4: Universal Topology

**Claim**: See `paper/manifest.yaml`::R6

**Runtime**: ~1 minute

This notebook verifies universal topology claim: n = 2.00 ± 0.05 across all systems, with Influenza exception.

In [ ]:
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Load manifest
manifest_path = Path('../../manifest.yaml')
manifest = yaml.safe_load(manifest_path.open())
result = manifest['results']['R6']  # R6: Universal n = 2.00 ± 0.05 topology

print(f"Verifying: {result['title']}")
print(f"Result ID: R6")
print(f"Category: topology")

## Load Canonical Data

In [ ]:
# Load phylogenetic tree and viral results (v2 schema)
tree_path = Path('../../validation/phylogenetic/results/tree_kappa_estimates.yaml')
viral_path = Path('../../validation/viral/results/fifteen_virus_sweeps.yaml')

if not tree_path.exists():
    raise FileNotFoundError(f"Tree results not found: {tree_path}")

trees = yaml.safe_load(tree_path.open())
domain_trees = trees['domain_trees']  # v2: list of domain-level embeddings

print(f"Loaded {len(domain_trees)} domain-level trees")
print(f"Sources: {set(t['source'] for t in domain_trees)}")
print()
print(f"{'Domain':<12} {'tips':>8}  {'kappa':>8}  {'n_back':>8}  stress")
for t in domain_trees:
    print(f"  {t['domain']:<10} {t['tips']:>8}  {t['kappa']:>8.2f}  {t['n_backsolved']:>8.2f}  {t['stress_normalized']:.3f}")

# Viral families (§5, 15 families)
viral = yaml.safe_load(viral_path.open())
viral_families = viral['families']
print(f"\n{len(viral_families)} viral families loaded (n back-solved mean = "
      f"{np.mean([f['n_backsolved'] for f in viral_families]):.3f})")

# Build combined topology DataFrame
rows = []
for t in domain_trees:
    rows.append({'system': t['domain'], 'source': t['source'].split(',')[0], 'n': t['n_backsolved'], 'kappa': t['kappa']})
for f in viral_families:
    rows.append({'system': f['name'], 'source': 'viral', 'n': f['n_backsolved'], 'kappa': f['kappa']})
topology_df = pd.DataFrame(rows)
print(f"\nCombined: {len(topology_df)} systems")


## Verify Claims

In [ ]:
# =============================================================================
# Check 1: All systems converge to n = 2 (universal invariant per paper §2)
# =============================================================================
print("Check 1: Universal invariant n = 2")

n_values = topology_df['n'].values
n_mean = float(n_values.mean())
n_std = float(n_values.std())

expected_mean = 2.00
expected_uncertainty = 0.05

# Paper claim: "n = 2.00 ± 0.05 across every system tested"
passed_1 = abs(n_mean - expected_mean) <= 2 * expected_uncertainty and n_std < 0.15
print(f"  Mean n = {n_mean:.3f}")
print(f"  Std n  = {n_std:.3f}")
print(f"  Expected: {expected_mean:.2f} +/- {expected_uncertainty:.2f}")
print(f"  Status: {'PASS' if passed_1 else 'FAIL'}")

print(f"\n  Per-system values:")
for _, row in topology_df.iterrows():
    print(f"    {row['system']:20s} ({row['source']:10s}): n = {row['n']:.2f}, kappa = {row['kappa']:.2f}")

# =============================================================================
# Check 2: Intra-domain κ matches state-equation prediction at scale-appropriate h
# (paper §4.2 Table 3; replaces the legacy bootstrap-CI-contains-1.247 check)
# =============================================================================
print("\nCheck 2: Intra-domain κ consistent with scale-dependence summary")

domain_rows = topology_df[topology_df['source'].isin(['Li et al. 2021', 'GTDB r220 archaeal species tree', 'GTDB r220 bacterial species tree'])]
canonical = {'Fungi': (3.0, 0.5), 'Archaea': (12.7, 2.0), 'Bacteria': (16.4, 2.0)}
passed_2 = True
for _, row in domain_rows.iterrows():
    target, tol = canonical.get(row['system'], (None, None))
    if target is None:
        continue
    ok = abs(row['kappa'] - target) <= tol
    passed_2 = passed_2 and ok
    print(f"  {row['system']:10s}: measured κ = {row['kappa']:.2f}, canonical = {target}, within ±{tol}: {ok}")

print(f"  Status: {'PASS' if passed_2 else 'FAIL'}")

# =============================================================================
# Compile
# =============================================================================
verified_checks = [
    {'name': 'all_systems_n_equals_2', 'expected': 'mean n = 2.00 ± 0.05',
     'passed': bool(passed_1), 'value': {'mean': n_mean, 'std': n_std}},
    {'name': 'intra_domain_kappa_matches_v2', 'expected': 'fungi 3.0, archaea 12.7, bacteria 16.4',
     'passed': bool(passed_2)},
]

all_passed = all(c['passed'] for c in verified_checks)
print(f"\n{'='*60}")
print(f"Overall: {'PASS' if all_passed else 'FAIL'}  ({sum(c['passed'] for c in verified_checks)}/2 checks)")
print(f"{'='*60}")


## Update Results

In [ ]:
try:
    results_path = Path('../../results.yaml')

    if results_path.exists():
        data = yaml.safe_load(results_path.open()) or {}
        results = data.get('results', {})
    else:
        results = {}

    if 'R6' not in results:
        results['R6'] = {}

    results['R6']['verified'] = all_passed
    results['R6']['verification_date'] = datetime.now().isoformat()
    results['R6']['checks'] = verified_checks

    output = {'results': results}
    with results_path.open('w') as f:
        yaml.dump(output, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

    print(f"Results updated in {results_path}")
    print(f"  Verified: {all_passed}")
    print(f"  Date: {results['R6']['verification_date']}")
except Exception as e:
    print(f"Note: results.yaml update skipped ({e})")